In [ ]:
import pandas as pd

# Cargar datos históricos
df_matches = pd.read_csv("../data/raw/WorldCupMatches.csv")

# Calcular victorias históricas por equipo
historico = []
for equipo in df_matches["Home Team Name"].unique():
    # Partidos como local
    local = df_matches[df_matches["Home Team Name"] == equipo]
    victorias_local = (local["Home Team Goals"] > local["Away Team Goals"]).sum()
    
    # Partidos como visitante
    visitante = df_matches[df_matches["Away Team Name"] == equipo]
    victorias_visitante = (visitante["Away Team Goals"] > visitante["Home Team Goals"]).sum()
    
    total_victorias = victorias_local + victorias_visitante
    total_partidos = len(local) + len(visitante)
    mundiales = df_matches[df_matches["Home Team Name"] == equipo]["Year"].nunique()
    
    historico.append({
        "country_name": equipo,
        "victorias_historicas": total_victorias,
        "mundiales_anteriores": mundiales,
        "victorias_pct_historico": round(100 * total_victorias / total_partidos, 1) if total_partidos > 0 else 0
    })

df_historico = pd.DataFrame(historico)

In [ ]:
import pandas as pd
import numpy as np
import joblib

# load model and trained scaler
model = joblib.load("../outputs/final_model.pkl")
scaler = joblib.load("../outputs/scaler.pkl")

# load actually data of nations 2026
# (You get them from football-data.org or build them manually using FBref.)
df_2026 = pd.read_csv("../data/raw/rankings_2026.csv")

FEATURES = [
    "fifa_rank", "group_stage_goals", "goals_conceded",
    "goal_differential", "victorias_historicas", 
    "mundiales_anteriores", "victorias_pct_historico"
]

X_2026 = df_2026[FEATURES]
X_2026_scaled = scaler.transform(X_2026)

In [ ]:
# Generate predictions
probabilities = model.predict_proba(X_2026_scaled)[:, 1]
predictions = model.predict(X_2026_scaled)

# Create results table
results = pd.DataFrame({
    "seleccion": df_2026["pais"],
    "confederacion": df_2026["confederacion"],
    "ranking_fifa": df_2026["ranking_fifa"],
    "probabilidad_semis": (probabilities * 100).round(1),
    "prediccion": predictions.map({1: "✅ Reaches the semifinals", 0: "❌ They won't make it to the semifinals"})
}).sort_values("probabilidad_semis", ascending=False)

results["ranking_prediccion"] = range(1, len(results) + 1)

print("=" * 60)
print("PREDICCIONES MUNDIALES 2026 — TOP 10")
print("=" * 60)
print(results.head(10).to_string(index=False))